## Desafio 2 - Leave-One-Instance-Out Bias-Free Protocol

Os resultados estão bastante suspeitos. Voces disseram inicialmente que o leave one instance out tinha resultados superiores aos reportados por Fernandes, mas no artigo estão bem piores (só o LOF que está minimamente aceitável). Eu não estou confiando nesses resultados. Será preciso verificar e provavelmente refazer.

Andei pensando por aqui e acho que tem uma boa estrategia para ser seguida. teremos que refazer os experimentos:

Devemos comparar as metodologias do Ricardo (valores default de hiperparametros) e do Fernandes (valor ajustados nos dados simulados), que separam treino e teste por janelamento da mesma instancia apenas, com nossas duas metodologia: que separa por instancia e que separa  por poco, e ver os impactos. É esperado que a separação por instancia reduza o desempenho dos modelos e que a separação por poço reduza ainda mais.

Os experimentos anteriores devem revelar que há perda de desempenho por conta do bias de similaridade para o problema geral de detecção de falha. O próximo passo, seria computar os resultados para cada falha, para verificar quais falhas são mais suscetíveis ao bias de similaridade.

-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

Foco é na metodologia de experimentação One Class

- Leave-one-out de instancias - Conferir resultados alcançados (Por que melhor de Wander (LOF) tem resultado muito pior)

- Fazer Leave-one-out de poços


In [1]:
from datetime import datetime
start_time=datetime.now()
import pandas as pd
import numpy as np
import logging
import warnings
from time import time
from pathlib import Path
from tsfresh.feature_extraction import extract_features
from tsfresh.utilities.dataframe_functions import impute
from tsfresh.feature_extraction import MinimalFCParameters
from sklearn.svm import OneClassSVM
from sklearn.covariance import EllipticEnvelope
from sklearn.neighbors import LocalOutlierFactor
from sklearn.ensemble import IsolationForest
from sklearn.dummy import DummyClassifier
from sklearn import preprocessing
from sklearn.metrics import precision_recall_fscore_support
logging.getLogger('tsfresh').setLevel(logging.ERROR)
warnings.simplefilter(action='ignore')
data_path=Path('./','data')
random_state=1
np.random.seed(random_state)
vars=['P-PDG','P-TPT','T-TPT','P-MON-CKP','T-JUS-CKP','P-JUS-CKGL','T-JUS-CKGL','QGL']
columns=['timestamp']+vars+['class']
normal_class_code=0
abnormal_classes_codes=[1,2,5,6,7,8]
sample_size=3*60
min_normal_period_size=20*60
max_nan_percent=0.1
std_vars_min=0.01
disable_progressbar=True
df_fc_p=MinimalFCParameters()
df_fc_p.pop('absolute_maximum')
df_fc_p.pop('sum_values')
df_fc_p.pop('length')
def class_and_file_generator(data_path,real=False,simulated=False,drawn=False):
    for class_path in data_path.iterdir():
        if class_path.is_dir():
            class_code=int(class_path.stem)
            for instance_path in class_path.iterdir():
                if instance_path.suffix=='.csv':
                    if(real and(not instance_path.stem.startswith('SIMULATED'))and(not instance_path.stem.startswith('DRAWN'))):
                        yield class_code,instance_path
def load_instance(instance_path):
    try:
        well,instance_id=instance_path.stem.split('_')
        df=pd.read_csv(instance_path,sep=',',header=0)
        assert(df.columns==columns).all(),f'Colunas inválidas: {str(df.columns.tolist())}'
        return well,df
    except Exception as e:
        raise Exception(f'Erro ao ler arquivo {instance_path}: {e}')
def extrair_janelas_stride(df,sample_id_offset=0):
    df_samples=pd.DataFrame()
    labels=[]
    n=len(df)
    n_janelas=n//sample_size
    sample_id=sample_id_offset
    df_vars=df.drop('class',axis=1).fillna(0)
    for j in range(n_janelas):
        ini=j*sample_size
        janela=df_vars.iloc[ini:ini+sample_size,:].copy()
        janela.insert(loc=0,column='id',value=sample_id)
        df_samples=pd.concat([df_samples,janela],ignore_index=True)
        classe_janela=df['class'].iloc[ini:ini+sample_size]
        if(classe_janela!=0).any():labels.append(-1)
        else:labels.append(1)
        sample_id+=1
    return df_samples,labels,sample_id
def train_test_calc_scores(X_train,y_train,X_test,y_test,scores,clfs,poco,class_code,instance_path):
    X_train.reset_index(inplace=True,drop=True)
    X_test.reset_index(inplace=True,drop=True)
    for clf_name,clf in clfs.items():
        t0=time()
        clf.fit(X_train,y_train)
        t_train=time()-t0
        t0=time()
        y_pred=clf.predict(X_test)
        t_test=time()-t0
        ret=precision_recall_fscore_support(y_test,y_pred,average='micro')
        p,r,f1,_=ret
        scores=pd.concat([scores,pd.DataFrame([{'INSTANCIA':str(instance_path),'POCO':poco,'CLASSE':class_code,'CLASSIFICADOR':clf_name,'PRECISAO':p,'REVOGACAO':r,'F1':f1,'TREINAMENTO [s]':t_train,'TESTE [s]':t_test}])],ignore_index=True)
    return scores
all_instances=pd.DataFrame(class_and_file_generator(data_path,real=True,simulated=False,drawn=False),columns=['class_code','instance_path'])
test_instances=all_instances.loc[all_instances['class_code'].isin(abnormal_classes_codes)].reset_index(drop=True)
normal_instances=all_instances.loc[all_instances['class_code']==normal_class_code].reset_index(drop=True)
scores=pd.DataFrame()
ignored_instances=0
used_instances=0
for i,row in test_instances.iterrows():
    class_code,instance_path=row
    print(f'Instância {i+1}: {instance_path}')
    poco_test,df_test=load_instance(instance_path)
    normal_period_size=(df_test['class']==0).sum()
    if normal_period_size<min_normal_period_size:
        ignored_instances+=1
        print(f'\tIgnorado por normalidade insuficiente ({normal_period_size} linhas)')
        continue
    used_instances+=1
    df_samples_test,y_test,_=extrair_janelas_stride(df_test,sample_id_offset=0)
    df_samples_train=pd.DataFrame()
    y_train=[]
    sample_id_offset=0
    for _,norm_row in normal_instances.iterrows():
        _,norm_path=norm_row
        # LOO por instância: exclui apenas a instância testada
        # Como instâncias de treino são sempre normais (class==0)
        # e a instância testada é sempre anômala, nenhuma exclusão
        # é necessária na prática — mas mantemos a verificação por segurança
        if norm_path==instance_path:
            continue
        norm_period_size=(pd.read_csv(norm_path,sep=',',header=0)['class']==0).sum()
        if norm_period_size<min_normal_period_size:
            continue
        _,df_norm=load_instance(norm_path)
        df_s,labels,sample_id_offset=extrair_janelas_stride(df_norm,sample_id_offset=sample_id_offset)
        df_samples_train=pd.concat([df_samples_train,df_s],ignore_index=True)
        y_train.extend(labels)
    if len(y_train)==0:
        print(f'\tIgnorado: sem instâncias de treino disponíveis para {instance_path}')
        ignored_instances+=1
        used_instances-=1
        continue
    y_train=np.array(y_train)
    y_test=np.array(y_test)
    good_vars=np.isnan(df_samples_train[vars]).mean(0)<=max_nan_percent
    std_vars=np.nanstd(df_samples_train[vars],0)
    good_vars&=(std_vars>std_vars_min)
    good_vars=list(good_vars.index[good_vars])
    bad_vars=list(set(vars)-set(good_vars))
    df_samples_train.drop(columns=bad_vars,inplace=True,errors='ignore')
    df_samples_test.drop(columns=bad_vars,inplace=True,errors='ignore')
    scaler=preprocessing.StandardScaler()
    df_samples_train[good_vars]=scaler.fit_transform(df_samples_train[good_vars]).astype('float32')
    df_samples_test[good_vars]=scaler.transform(df_samples_test[good_vars]).astype('float32')
    X_train=extract_features(df_samples_train,column_id='id',column_sort='timestamp',default_fc_parameters=df_fc_p,impute_function=impute,n_jobs=0,disable_progressbar=disable_progressbar)
    X_train=X_train.reset_index(drop=True)
    X_test=extract_features(df_samples_test,column_id='id',column_sort='timestamp',default_fc_parameters=df_fc_p,impute_function=impute,n_jobs=0,disable_progressbar=disable_progressbar)
    X_test=X_test.reset_index(drop=True)
    clfs={'Local Outlier Factor':LocalOutlierFactor(n_neighbors=20,algorithm='auto',metric='euclidean',contamination='auto',novelty=True),
          'Floresta de Isolamento':IsolationForest(n_estimators=200,max_samples=0.75,contamination=0.1,max_features=1.0,bootstrap=False,random_state=1,verbose=0),
          'Envelope Eliptico MCD':EllipticEnvelope(contamination=0.001,assume_centered=False,support_fraction=0.99,random_state=1),
          'One Class SVM':OneClassSVM(kernel='sigmoid',gamma=0.01,nu=0.001),
          'One Class SVM (benchmark Vargas)':OneClassSVM(kernel='sigmoid',gamma='scale',nu=0.5),
          'Floresta de Isolamento (benchmark Vargas)':IsolationForest(n_jobs=None,contamination=0.000001,random_state=random_state),
          'Dummy':DummyClassifier(strategy='constant',constant=1)}
    scores=train_test_calc_scores(X_train,y_train,X_test,y_test,scores,clfs,poco_test,class_code,instance_path)
print(f'Número de instâncias utilizadas: {used_instances}')
print(f'Número de instâncias ignoradas: {ignored_instances}')
scores.to_csv(r'./results/loo_instancia_scores.csv')
mean_score_table=scores.groupby('CLASSIFICADOR')[['PRECISAO','REVOGACAO','F1','TREINAMENTO [s]','TESTE [s]']].mean().sort_values(by=['F1'],ascending=False)
mean_score_table.to_csv(r'./results/loo_instancia_scores_medias.csv')
print(mean_score_table)
std_score_table=scores.groupby('CLASSIFICADOR')[['PRECISAO','REVOGACAO','F1','TREINAMENTO [s]','TESTE [s]']].std().sort_values(by=['F1'],ascending=True)
std_score_table.to_csv(r'./results/loo_instancia_scores_desvios_padrao.csv')
print(std_score_table)
print(f'Tempo total de execução (hh:mm:ss.ms): {datetime.now()-start_time}')

C:\Users\UQBU\AppData\Local\miniforge3\envs\wander\lib\site-packages\sklearn\linear_model\least_angle.py:30: DeprecationWarning: `np.float` is a deprecated alias for the builtin `float`. To silence this warning, use `float` by itself. Doing this will not modify any behavior and is safe. If you specifically wanted the numpy scalar type, use `np.float64` here.
Deprecated in NumPy 1.20; for more details and guidance: https://numpy.org/devdocs/release/1.20.0-notes.html#deprecations
  method='lar', copy_X=True, eps=np.finfo(np.float).eps,
C:\Users\UQBU\AppData\Local\miniforge3\envs\wander\lib\site-packages\sklearn\linear_model\least_angle.py:167: DeprecationWarning: `np.float` is a deprecated alias for the builtin `float`. To silence this warning, use `float` by itself. Doing this will not modify any behavior and is safe. If you specifically wanted the numpy scalar type, use `np.float64` here.
Deprecated in NumPy 1.20; for more details and guidance: https://numpy.org/devdocs/release/1.20.0-

Instância 1: data\1\WELL-00001_20140124213136.csv
	Ignorado por normalidade insuficiente (959 linhas)
Instância 2: data\1\WELL-00002_20140126200050.csv
	Ignorado por normalidade insuficiente (1138 linhas)
Instância 3: data\1\WELL-00006_20170801063614.csv
Instância 4: data\1\WELL-00006_20170802123000.csv
Instância 5: data\1\WELL-00006_20180618060245.csv
Instância 6: data\2\WELL-00002_20131104014101.csv
Instância 7: data\2\WELL-00003_20141122214325.csv
Instância 8: data\2\WELL-00003_20170728150240.csv
Instância 9: data\2\WELL-00003_20180206182917.csv
	Ignorado por normalidade insuficiente (586 linhas)
Instância 10: data\2\WELL-00009_20170313160804.csv
Instância 11: data\2\WELL-00010_20171218200131.csv
Instância 12: data\2\WELL-00011_20140515110134.csv
Instância 13: data\2\WELL-00011_20140530100015.csv
	Ignorado por normalidade insuficiente (482 linhas)
Instância 14: data\2\WELL-00011_20140606230115.csv
Instância 15: data\2\WELL-00011_20140720120102.csv
Instância 16: data\2\WELL-00011_201